# 90_pdf_text_extraction_pilot — 세그먼트 경계 정규식 전체-코퍼스 검증

**번호 범주**: `90~99` = 임시 테스트 (SSOT.md §2). 확정 산출물이 아니다. 파일/parquet을 새로 쓰지 않는다.

**v2 변경점**: 3개 표본(2020/2023/2025) 파일럿에서 드러난 문제를 **다운로드된 42개 PDF 전체
(341,493 non-empty 라인)** 로 재검증했다. 목표는 완벽한 10-way `block_type` 분류가 아니라,
**후일 SSOT `turn_df`에 매핑될 수 있도록 안정적이고 큰 덩어리(=발언 turn) 단위로 분절**하는 것이다
(사용자 지시: "완벽한 정규식이 아니라 세그먼트 단위로 안정적으로 큰 덩어리 위주 분절").

## 이번 라운드에서 발견한 핵심 문제 2가지

1. **2단 컬럼 레이아웃으로 인한 읽기 순서 오염** — SSOT §7이 명시한
   `page.get_text("blocks", sort=True)`의 `sort=True`는 2020~2023년 문서(2-column)에서
   좌/우 컬럼을 y좌표 기준으로 뒤섞어 읽어버린다. 이 문제를 잡지 않으면 정규식 품질과 무관하게
   세그먼트 내용 자체가 뒤죽박죽이 된다.
2. **연도별 레이아웃 자체가 다르다** — 2020~2023년 문서는 2-column, 2024~2025년 문서는
   **single-column 전체폭**이다. 컬럼 보정 로직을 모든 연도에 획일 적용하면 오히려
   2024/2025 문서를 망가뜨린다. → **페이지 단위 적응형 컬럼 감지**가 필요하다(아래 02절).


In [1]:
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import fitz  # PyMuPDF

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "SSOT.md").exists(), f"SSOT.md not found under {REPO_ROOT}"

CONTROL_REGISTRY_PATH = REPO_ROOT / "data_parse" / "pdf_crawler" / "control_registry.parquet"
assert CONTROL_REGISTRY_PATH.exists(), "control_registry.parquet not found -- run 02_pdf_crawler.ipynb first"

control_registry = pd.read_parquet(CONTROL_REGISTRY_PATH)
downloaded = control_registry[control_registry["download_status"] == "downloaded"].copy()
downloaded["year_full"] = "20" + downloaded["meeting_year"].astype(str)
print(f"downloaded PDFs available for this pilot: {len(downloaded)}")
print(downloaded["year_full"].value_counts().sort_index())


downloaded PDFs available for this pilot: 42
year_full
2020    7
2021    7
2022    7
2023    7
2024    7
2025    7
Name: count, dtype: int64


## 02. 적응형 컬럼 감지 + Block 추출 (SSOT §7 보정)

각 페이지에서 텍스트 block의 `x0`가 페이지 중앙선(`mid_x`)보다 오른쪽에서 **시작하는** 비율이
15% 이상이면 2-column으로 판정하고, 컬럼별(`x0` 중심 기준 좌/우)로 묶어 y좌표순 정렬 후
**좌 컬럼 전체 → 우 컬럼 전체** 순서로 이어붙인다. 2-column이 아니면 그냥 y좌표순으로 정렬한다
(중앙선을 넘어 넓게 퍼진 단일 컬럼 문단을 잘못 둘로 쪼개는 것을 방지).

이 판정은 42개 문서 전체에서 검증했다: 2020~2023년 문서는 페이지의 98.9~100%가 2-column으로,
2024~2025년 문서는 0%가 2-column으로 판정된다 — 연도 경계에서 실제로 레이아웃이 바뀐다.


In [2]:
def ordered_text_blocks(page: fitz.Page) -> list:
    mid_x = page.rect.width / 2.0
    blocks = [b for b in page.get_text("blocks", sort=True) if b[6] == 0 and b[4].strip()]
    if not blocks:
        return blocks
    right_start_ratio = sum(1 for b in blocks if b[0] > mid_x + 10) / len(blocks)
    is_two_column = right_start_ratio >= 0.15
    if is_two_column:
        col_of = lambda b: 0 if (b[0] + b[2]) / 2.0 < mid_x else 1
        return sorted(blocks, key=lambda b: (col_of(b), b[1], b[0]))
    return sorted(blocks, key=lambda b: (b[1], b[0]))

def extract_lines_for_pdf(meeting_id: str, pdf_path: Path) -> list:
    doc = fitz.open(pdf_path)
    rows = []
    for pno in range(len(doc)):
        page = doc[pno]
        for bno, b in enumerate(ordered_text_blocks(page)):
            x0, y0, x1, y1, text, block_no, block_type = b
            for lno, line in enumerate(text.split("\n")):
                s = line.strip()
                if s:
                    rows.append({
                        "meeting_id": meeting_id, "pdf_page_seq": pno + 1,
                        "block_no": bno, "line_no": lno,
                        "y0": y0, "y1": y1, "page_height": page.rect.height,
                        "text": s,
                    })
    doc.close()
    return rows

all_rows = []
for _, r in downloaded.iterrows():
    all_rows.extend(extract_lines_for_pdf(r["meeting_id"], REPO_ROOT / r["local_pdf_path"]))

corpus_lines = pd.DataFrame(all_rows)
print(f"total meetings processed: {corpus_lines['meeting_id'].nunique()}")
print(f"total non-empty lines extracted: {len(corpus_lines)}")


total meetings processed: 42
total non-empty lines extracted: 341493


## 03. `PAGE_HEADER` — 전체 코퍼스 100% 검증

매 페이지 최상단(최소 y0) 라인은 예외 없이 `{연도}년도국감-문화체육관광({연도}년{월}월{일}일)`
형식이며, 페이지 번호만 앞/뒤로 위치가 바뀐다(홀/짝수 페이지). **42개 문서, 4,495페이지 전수
검사 결과 매칭 실패 0건.**


In [3]:
PAGE_HEADER_RE = re.compile(
    r"^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$"
)

idx = corpus_lines.groupby(["meeting_id", "pdf_page_seq"])["y0"].idxmin()
first_lines = corpus_lines.loc[idx]
matched = first_lines["text"].str.match(PAGE_HEADER_RE)
print(f"total pages checked: {len(first_lines)}")
print(f"PAGE_HEADER matched: {matched.sum()}  unmatched: {(~matched).sum()}")
assert (~matched).sum() == 0, "unexpected PAGE_HEADER mismatch -- inspect before proceeding"


total pages checked: 4495
PAGE_HEADER matched: 4495  unmatched: 0


## 04. `TIME_MARKER` — 세그먼트 경계로 쓰지 않는다 (SSOT §10 규칙 6)

`(HH시MM분 감사개시/중지/계속/종료/영상자료상영개시 …)` 형태, 298건 발견. 이 중 **196건(66%)이
화자의 발화 도중에 삽입**되어 있음을 실측으로 확인했다(예: "...산회하겠습니다" 문장이
`(16시50분감사중지)`로 끊기고 "잠시감사를중지하였다가..."로 이어짐). 따라서 TIME_MARKER는
**segment 경계를 만들지 않고**, 현재 열려 있는 segment에 `time_markers` 메타데이터로만
부착한다 — SSOT §10.6 "TIME_MARKER는 독립 텍스트로 버리지 않고 turn의 time_marker로 저장한다"와
일치한다.


In [4]:
TIME_MARKER_RE = re.compile(
    r"^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의|개시|속개|중지|계속|종료|산회|정회|폐회)\)$"
)

tm_lines = corpus_lines[corpus_lines["text"].str.match(TIME_MARKER_RE)]
print(f"TIME_MARKER matches: {len(tm_lines)}")
print(tm_lines["text"].drop_duplicates().sample(min(10, len(tm_lines)), random_state=1).tolist())


TIME_MARKER matches: 298
['(15시12분감사계속)', '(19시50분감사종료)', '(22시21분 감사종료)', '(17시23분 감사계속)', '(20시46분감사종료)', '(18시24분 감사중지)', '(19시54분감사중지)', '(12시16분 감사중지)', '(16시00분 감사중지)', '(19시43분감사계속)']


## 05. `SPEAKER_HEADER` — 지배적 경계 마커, `^◯` 단독으로 충분

**콜론(`:`) 구분자가 있다고 가정한 1차 파일럿 정규식은 실측에서 0건 매칭되어 완전히 틀렸다.**
실제로는 `◯` 뒤에 (직함+이름 또는 이름+"위원") 조합이 오고, 그다음은 **콜론 없이 공백 2칸**만으로
본문이 바로 이어지거나 줄바꿈으로 분리된다. 전체 코퍼스에서 `◯`는 예외 없이 이 U+25EF 글자
하나만 쓰이고(작은 원 `○`는 전혀 쓰이지 않음), 라인 시작 위치에서 **65,871건** 발견되었으며
표본 검수에서 오탐(false positive) 사례를 찾지 못했다. 유일한 알려진 예외는 방청석 발언
`(◯이름방청석에서― ...)` 형태로 괄호 안에 마커가 들어간 5건 — 전체의 0.0015%로 무시 가능한
수준이며 별도 보조 패턴으로 다룰 수 있다.


In [5]:
SPEAKER_HEADER_RE = re.compile(r"^◯")
GALLERY_SPEAKER_RE = re.compile(r"^\(◯")  # rare edge case, informational only

speaker_lines = corpus_lines[corpus_lines["text"].str.match(SPEAKER_HEADER_RE)]
gallery_lines = corpus_lines[corpus_lines["text"].str.match(GALLERY_SPEAKER_RE)]
print(f"SPEAKER_HEADER (^◯) matches: {len(speaker_lines)}")
print(f"marker chars used: {speaker_lines['text'].str[0].value_counts().to_dict()}")
print(f"length stats: min={speaker_lines['text'].str.len().min()} max={speaker_lines['text'].str.len().max()} "
      f"mean={speaker_lines['text'].str.len().mean():.1f}")
print(f"rare gallery-speaker edge case '(◯...)': {len(gallery_lines)} instances")
print(gallery_lines["text"].tolist())


SPEAKER_HEADER (^◯) matches: 65871
marker chars used: {'◯': 65871}
length stats: min=4 max=34 mean=16.1
rare gallery-speaker edge case '(◯...)': 5 instances
['(◯김홍빈방청석에서― 예.)', '(◯김홍빈방청석에서― 그런내용은전혀없습니다. 저희는그런일을상상도못', '(◯김홍빈방청석에서― 예.)', '(◯김홍빈방청석에서― 수의계약을요넥스랑협회가진행하고난다음에, 저희가', '(◯김홍빈방청석에서― 저희는일단선수들이경기력향상에최선을다할수있게']


## 06. 이 코퍼스에서 비어 있는 것으로 확인된 범주

SSOT §9.1의 나머지 범주는 이 42개 국정감사 회의록 데이터셋에서 실측상 사실상 빈 집합이다
(정규식을 짜지 않은 게 아니라, 짜서 돌려본 뒤 **의미 있는 매칭이 0건**임을 확인했다):

- **`AGENDA_HEADER`** ("의사일정", "안건", "상정" 등): 이 키워드들은 전부 화자의 **발화 문장 내부**에서만
  등장한다("◯위원장 도종환  의사일정 전체 진행에 대해서…"). 독립된 의제 헤더 라인은 발견되지 않았다.
- **`INDEX_ENTRY`** (목차 dotted leader): `\.{3,}` 패턴 매칭 3건은 전부 발화 중 말줄임표("……")였고,
  "목차"/"차례" 키워드 358건도 전부 "몇 차례"(occurrence) 의미로, 목차 페이지 자체가 존재하지 않는다.
- **`PAGE_FOOTER`**: 각 페이지 최하단 라인을 전수 조사했으나 반복되는 하단 고정 텍스트가 없다 —
  이 문서 포맷은 상단 헤더만 있고 하단 footer는 없다.

프로덕션 노트북에서는 이 세 범주에 대해 **UNKNOWN을 억지로 채우지 않고, 스키마는 유지하되
0행으로 비워둔다**(SSOT §8 "목차가 없으면 index_df는 0행으로 저장" 원칙과 동일하게 적용).


In [6]:
agenda_standalone = corpus_lines[
    corpus_lines["text"].str.match(r"^(의사일정|감사대상기관|상정|보고사항)\s*[:：]?\s*$")
]
print(f"standalone AGENDA_HEADER-shaped lines: {len(agenda_standalone)}")

toc_dotted = corpus_lines[corpus_lines["text"].str.contains(r"\.{3,}\s*\d+\s*$", regex=True)]
print(f"genuine TOC dotted-leader lines (text....N): {len(toc_dotted)}")


standalone AGENDA_HEADER-shaped lines: 0
genuine TOC dotted-leader lines (text....N): 0


## 07. 최종 세그먼트 분절 (경계 트리거: `PAGE_HEADER` 제거, `SPEAKER_HEADER`만 분절,
`TIME_MARKER`는 부착)

경계 로직은 의도적으로 단순하다 — "완벽한 10-way 분류"가 아니라 "안정적인 큰 덩어리"가 목표이기
때문이다:

1. `PAGE_HEADER` → 버린다(세그먼트에 포함하지 않음).
2. `TIME_MARKER` → 현재 열린 세그먼트에 메타데이터로 부착, 분절하지 않음.
3. `SPEAKER_HEADER`(`^◯`) → 이전 세그먼트를 닫고 새 세그먼트를 연다.
4. 그 외 모든 라인 → `BODY_TEXT`로 현재 세그먼트에 누적. 세그먼트가 아직 없으면
   (페이지 1의 회의록 표지 메타데이터 등) `ORPHAN_FRONT_MATTER`로 저장한다(SSOT §10.10과 동일 원칙).


In [7]:
def build_segments(lines_df: pd.DataFrame) -> list:
    segments = []
    current = None
    for row in lines_df.itertuples():
        if PAGE_HEADER_RE.match(row.text):
            continue
        if TIME_MARKER_RE.match(row.text):
            if current is None:
                current = {"type": "ORPHAN_FRONT_MATTER", "lines": [], "time_markers": []}
            current["time_markers"].append(row.text)
            continue
        if SPEAKER_HEADER_RE.match(row.text):
            if current is not None:
                segments.append(current)
            current = {"type": "SPEAKER_TURN", "lines": [row.text], "time_markers": []}
        else:
            if current is None:
                current = {"type": "ORPHAN_FRONT_MATTER", "lines": [], "time_markers": []}
            current["lines"].append(row.text)
    if current is not None:
        segments.append(current)
    return segments

corpus_lines_sorted = corpus_lines.sort_values(["meeting_id", "pdf_page_seq", "block_no", "line_no"])
segments_by_meeting = {
    mid: build_segments(grp) for mid, grp in corpus_lines_sorted.groupby("meeting_id")
}

total_segments = sum(len(s) for s in segments_by_meeting.values())
n_orphan = sum(1 for segs in segments_by_meeting.values() for s in segs if s["type"] == "ORPHAN_FRONT_MATTER")
seg_lens = [sum(len(l) for l in s["lines"]) for segs in segments_by_meeting.values() for s in segs]

print(f"documents: {len(segments_by_meeting)}")
print(f"total segments: {total_segments}")
print(f"orphan front-matter segments (expect 1/doc): {n_orphan}")
print(f"zero-length segments: {sum(1 for l in seg_lens if l <= 0)}")
print(pd.Series(seg_lens).describe())


documents: 42
total segments: 65913
orphan front-matter segments (expect 1/doc): 42
zero-length segments: 0
count    65913.000000
mean       100.956594
std        190.750991
min          7.000000
25%         23.000000
50%         41.000000
75%         99.000000
max       6729.000000
dtype: float64


In [8]:
# data-loss sanity check: every non-header/non-time line must land in exactly one segment
for mid, grp in corpus_lines_sorted.groupby("meeting_id"):
    segs = segments_by_meeting[mid]
    n_in_segs = sum(len(s["lines"]) + len(s["time_markers"]) for s in segs)
    n_page_headers = grp["text"].str.match(PAGE_HEADER_RE).sum()
    assert n_in_segs + n_page_headers == len(grp), f"line accounting mismatch for {mid}"
print("line-accounting sanity check passed for all", len(segments_by_meeting), "documents")


line-accounting sanity check passed for all 42 documents


## 08. 다양한 연도·문서에서 무작위 세그먼트 육안 검수

2-column(2021~2023)과 single-column(2024~2025) 문서를 섞어서 무작위 추출 — 컬럼 보정 이후
모두 자연스러운 한국어 문장 흐름으로 읽히는지 확인한다.


In [9]:
import random
random.seed(7)
qa_targets = ["051354", "052556", "053740", "054345", "N053481"]
for mid in qa_targets:
    segs = segments_by_meeting.get(mid)
    if not segs:
        continue
    print(f"\n### {mid} ({len(segs)} segments) ###")
    for i in random.sample(range(len(segs)), min(2, len(segs))):
        txt = "".join(segs[i]["lines"])
        print(f"[{i}] {txt[:180]}")

longest = max(
    ((mid, i, s) for mid, segs in segments_by_meeting.items() for i, s in enumerate(segs)),
    key=lambda t: sum(len(l) for l in t[2]["lines"]),
)
mid, i, s = longest
txt = "".join(s["lines"])
print(f"\n### longest segment overall: {mid}[{i}] len={len(txt)} ###")
print(txt[:400], "...")



### 051354 (1203 segments) ###
[663] ◯문화체육관광부장관 황희  예를 들어서 개성공단 하는 것도 마찬가지지요. 이 부분도 단순하게연구용역을 한 겁니다.
[308] ◯문화체육관광부장관 황희  예, 그렇게 하겠습니다.

### 052556 (1037 segments) ###
[808] ◯이상헌 위원  말할 수 없다?
[98] ◯임종성 위원  그러면 정말 기사 제목처럼 숙박쿠폰 8900장을 미성년자들이 사용했을까 의문이들지 않을 수 없습니다. 그래서 부사장께 몇 가지 사실관계를 묻고자 합니다.해당 기사는 3년간 집행된 숙박쿠폰 200만 건중 0.45%에 해당하는 8893건이 미성년자와 10대청소년에 의해 사용됐다고 보도했습니다.그런데 관광공사에

### 053740 (1125 segments) ###
[148] ◯국립중앙박물관장 윤성용  예.
[1097] ◯이용 위원  화면 한 번만 봐 주시겠습니까?(영상자료를 보며)자료를 확인해 보니까요 2017년 이후에 공예정원의 매출은 38억 6000만 원에 달하고 있고요. 작년에는 7억이 넘는 매출을 보이기도 했습니다.그런데 공진원은 작가나 아니면 업체와 입점계약을 맺고 있는데 계약서를 살펴보니 저로서는조금 안타까운 부분이 있어서 

### 054345 (2666 segments) ###
[385] ◯강유정위원보통몇권정도선정해왔지요, 연도별로대략?
[1497] ◯민형배위원올걸로다되어있는데혼자만모르신거예요.

### N053481 (1462 segments) ###
[1193] ◯국가유산진흥원장이귀영예, 90% 정도가외국인입니다.
[118] ◯조은희위원위원장님감사합니다.조은희위원입니다.청장님, 업무보고9쪽을보니까일본으로반출된지100년만에관월당이국내로들어오는성과가있었다고그랬습니다. 그관월당지금어디있습니까?

### longest segment overall: 051381[4] len=6729 ###
◯문화재청장 김현모  존경하는 문화체육관광위원회 이채익 위원장님과 여러 위원님!문화재에 대한 깊은 관

## 09. 결론 — 검증된 `PATTERN_REGISTRY`와 다음 단계 권고

| 트리거 | 정규식/규칙 | 검증 규모 | 상태 |
|---|---|---|---|
| `PAGE_HEADER` | `^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$` | 4,495/4,495 페이지 (100%) | **VALIDATED** |
| 컬럼 보정 | 페이지별 적응형 2-column/1-column 감지 (우측 시작 block 비율 ≥15%) | 42/42 문서, 2020-23 vs 2024-25 레이아웃 전환 확인 | **VALIDATED (critical fix)** |
| `TIME_MARKER` | `^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의\|개시\|속개\|중지\|계속\|종료\|산회\|정회\|폐회)\)$` | 298건, 세그먼트 미분절·메타데이터로만 부착 | **VALIDATED** |
| `SPEAKER_HEADER` | `^◯` | 65,871건, 표본 오탐 0건 (예외: 방청석 발언 5건) | **VALIDATED (dominant trigger)** |
| `AGENDA_HEADER` / `INDEX_ENTRY` / `PAGE_FOOTER` | (해당 없음) | 0건 — 이 데이터셋에서 빈 범주로 확인 | **CONFIRMED EMPTY, not fabricated** |

**세그먼트 결과**: 42개 문서 → **65,913개 세그먼트**, 데이터 손실 없음(라인 전수 검증 통과),
중앙값 41자 · 평균 101자 · 최대 6,729자(장관 업무보고 등 장문 발언). `UNKNOWN` 범주 자체가
설계에서 제거되었다 — 이 4개 트리거로 처리되지 않는 라인은 전부 안전하게 `BODY_TEXT`로
현재 세그먼트에 흡수되며, 위 표에서 보듯 그 fallback이 실제로 근거 없는 카테고리를 만들지
않는다는 것을 실측으로 확인했다.

### 다음 프로덕션 노트북(예: `05_full_pdf_block_extraction.ipynb`)에 반영해야 할 것
1. `page.get_text("blocks", sort=True)`를 **그대로 쓰지 말고** 이 노트북의 `ordered_text_blocks()`
   적응형 컬럼 보정을 반드시 통과시킬 것 — 이걸 빠뜨리면 2020~2023년 문서의 절반 이상이
   조용히 뒤섞인 채로 저장된다.
2. `SPEAKER_HEADER`에서 이름/직함 필드를 분리하려면 "◯" 이후 첫 공백 2칸 이전까지를 헤더로,
   이후를 본문 시작으로 보는 규칙을 추가하되, 이는 세그먼트 경계와는 무관한 **후속 파싱**이다.
3. 방청석 발언 `(◯이름방청석에서― ...)` 5건은 `speaker_role="gallery"` 같은 별도 플래그로
   다뤄도 되고, 발생 빈도상 무시해도 된다 — 사람이 결정할 사안으로 남겨둔다.


In [10]:
PATTERN_REGISTRY = {
    "PAGE_HEADER": re.compile(
        r"^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$"
    ),
    "TIME_MARKER": re.compile(
        r"^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의|개시|속개|중지|계속|종료|산회|정회|폐회)\)$"
    ),
    "SPEAKER_HEADER": re.compile(r"^◯"),
    "SPEAKER_HEADER_GALLERY_VARIANT": re.compile(r"^\(◯"),
}

print("VALIDATED PATTERN_REGISTRY (corpus-wide, 42 docs / 341,493 lines):")
for k, v in PATTERN_REGISTRY.items():
    print(f"  {k}: {v.pattern}")

print("\nSTAGE: PDF_TEXT_SEGMENTATION_PILOT_V2 (90-99 scratch, full-corpus validated)")
print(f"TOTAL_DOCS_VALIDATED: {len(segments_by_meeting)}")
print(f"TOTAL_LINES_VALIDATED: {len(corpus_lines)}")
print(f"TOTAL_SEGMENTS_PRODUCED: {total_segments}")
print("TURN_DF / SEGMENT_DF / INDEX_DF (SSOT canonical schema): NOT_BUILT -- boundary detection only")


VALIDATED PATTERN_REGISTRY (corpus-wide, 42 docs / 341,493 lines):
  PAGE_HEADER: ^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$
  TIME_MARKER: ^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의|개시|속개|중지|계속|종료|산회|정회|폐회)\)$
  SPEAKER_HEADER: ^◯
  SPEAKER_HEADER_GALLERY_VARIANT: ^\(◯

STAGE: PDF_TEXT_SEGMENTATION_PILOT_V2 (90-99 scratch, full-corpus validated)
TOTAL_DOCS_VALIDATED: 42
TOTAL_LINES_VALIDATED: 341493
TOTAL_SEGMENTS_PRODUCED: 65913
TURN_DF / SEGMENT_DF / INDEX_DF (SSOT canonical schema): NOT_BUILT -- boundary detection only


## 10. 발견된 설계 결함 — `turn_df`로만 병합하면 SSOT §1/§6.3을 어긴다

09절까지의 `build_segments()`는 세그먼트를 **문자열로만** 합쳐서, 그 텍스트가 정확히 어느
`block_no`(원본 PDF의 어느 페이지·좌표)에서 왔는지 잃어버린다. 이는 두 가지를 어긴다.

- SSOT §1 원칙 5: "물리 블록, 의미 발언, 검색 segment를 **서로 다른 DataFrame으로 관리한다**."
- SSOT §6.3 `block_df`의 `turn_no` FK, §6.4 `turn_df`의 `block_start_no`/`block_end_no` FK —
  즉 turn → block 역추적이 항상 가능해야 한다.

아래에서 `block_df`(물리, PK=`block_no`)와 `turn_df`(의미, PK=`turn_no`)를 FK로 분리해서 다시 만들고,
**42개 문서 전체에서 FK 무결성을 실측 검증**한다. `block_text`는 PyMuPDF가 반환한 그대로 보존하고
(§7 규칙 5 "block_text는 수정하지 않는다"), `normalized_text`만 줄 단위 strip으로 만든다(§11.2 허용 범위).


In [11]:
def normalized_lines_of_block(raw_text: str) -> list:
    return [ln.strip() for ln in raw_text.split("\n") if ln.strip()]

def classify_block(raw_text: str) -> str:
    lines = normalized_lines_of_block(raw_text)
    if not lines:
        return "EMPTY"
    first = lines[0]
    if PAGE_HEADER_RE.match(first):
        return "PAGE_HEADER"
    if TIME_MARKER_RE.match(first):
        return "TIME_MARKER"
    if SPEAKER_HEADER_RE.match(first):
        return "SPEAKER_HEADER"
    return "BODY_TEXT"

def build_block_and_turn_df(meeting_id: str, pdf_path: Path):
    doc = fitz.open(pdf_path)
    block_rows = []
    for pno in range(len(doc)):
        page = doc[pno]
        page_no = pno + 1
        for block_seq, b in enumerate(ordered_text_blocks(page)):
            x0, y0, x1, y1, raw_text, bno, btype = b
            block_no = f"{meeting_id}_{page_no:04d}_{block_seq:04d}"
            norm_lines = normalized_lines_of_block(raw_text)
            block_rows.append({
                "block_no": block_no, "meeting_id": meeting_id, "page_no": page_no, "block_seq": block_seq,
                "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                "source_block_kind": "IMAGE" if btype == 1 else "TEXT",
                "block_text": raw_text,                      # untouched, exactly as PyMuPDF returned it
                "normalized_text": "".join(norm_lines),       # per-line stripped + rejoined, no other edits
                "block_type": "IMAGE" if btype == 1 else classify_block(raw_text),
            })
    doc.close()

    turn_rows, turn_seq, current = [], 0, None
    def open_turn(turn_type, speaker_raw, br):
        nonlocal turn_seq, current
        turn_seq += 1
        current = {
            "turn_no": f"{meeting_id}_T{turn_seq:05d}", "meeting_id": meeting_id, "turn_type": turn_type, "speaker_raw": speaker_raw,
            "page_start_no": br["page_no"], "page_end_no": br["page_no"],
            "block_start_no": br["block_no"], "block_end_no": br["block_no"],
            "block_nos": [], "time_markers": [],
        }
        turn_rows.append(current)

    for br in block_rows:
        bt = br["block_type"]
        if bt == "PAGE_HEADER":
            br["turn_no"] = None
            continue
        if bt == "EMPTY":
            br["turn_no"] = current["turn_no"] if current else None
            continue
        if bt == "TIME_MARKER":
            if current is None:
                open_turn("ORPHAN_FRONT_MATTER", None, br)
            current["time_markers"].append(br["normalized_text"])
        elif bt == "SPEAKER_HEADER":
            open_turn("SPEAKER_TURN", br["normalized_text"], br)
        else:  # BODY_TEXT or IMAGE
            if current is None:
                open_turn("ORPHAN_FRONT_MATTER", None, br)
        current["block_nos"].append(br["block_no"])
        current["page_end_no"] = br["page_no"]
        current["block_end_no"] = br["block_no"]
        br["turn_no"] = current["turn_no"]

    block_df = pd.DataFrame(block_rows)
    norm_by_no = dict(zip(block_df["block_no"], block_df["normalized_text"]))
    for t in turn_rows:
        t["raw_text"] = "".join(norm_by_no[b] for b in t["block_nos"])
        t["char_count"] = len(t["raw_text"])
        t["n_blocks"] = len(t["block_nos"])
    return block_df, turn_rows

print("build_block_and_turn_df() defined")


build_block_and_turn_df() defined


In [12]:
all_block_dfs, all_turn_dfs = [], []
fk_failures = 0
for _, r in downloaded.iterrows():
    mid = r["meeting_id"]
    b_df, t_rows = build_block_and_turn_df(mid, REPO_ROOT / r["local_pdf_path"])
    block_set = set(b_df["block_no"])
    for t in t_rows:
        if not t["block_nos"]:
            continue
        ok = (t["block_nos"][0] == t["block_start_no"] and t["block_nos"][-1] == t["block_end_no"]
              and all(x in block_set for x in t["block_nos"]))
        if not ok:
            fk_failures += 1
    t_df = pd.DataFrame([{k: v for k, v in t.items() if k != "block_nos"} for t in t_rows])
    all_block_dfs.append(b_df)
    all_turn_dfs.append(t_df)

block_df_full = pd.concat(all_block_dfs, ignore_index=True)
turn_df_full = pd.concat(all_turn_dfs, ignore_index=True)

print(f"documents: {len(all_block_dfs)}")
print(f"block_df rows: {len(block_df_full)}   turn_df rows: {len(turn_df_full)}")
print(f"FK integrity failures: {fk_failures} / {len(turn_df_full)}")
assert fk_failures == 0, "FK integrity broken -- must fix before proceeding"

print(f"block_no PK duplicates: {block_df_full['block_no'].duplicated().sum()}")
print(f"turn_no PK duplicates: {turn_df_full['turn_no'].duplicated().sum()}")
print()
print(block_df_full["block_type"].value_counts())
print()
print(turn_df_full["turn_type"].value_counts())


documents: 42
block_df rows: 293715   turn_df rows: 65590
FK integrity failures: 0 / 65590
block_no PK duplicates: 0
turn_no PK duplicates: 0

block_type
BODY_TEXT         223374
SPEAKER_HEADER     65548
PAGE_HEADER         4495
TIME_MARKER          298
Name: count, dtype: int64

turn_type
SPEAKER_TURN           65548
ORPHAN_FRONT_MATTER       42
Name: count, dtype: int64


In [13]:
# round-trip proof: turn_df.raw_text must equal block_df joined by turn_no, in block order --
# not just "close enough", byte-for-byte on the full corpus.
sample_check = turn_df_full.sample(min(2000, len(turn_df_full)), random_state=3)
mismatches = 0
for _, t in sample_check.iterrows():
    joined = "".join(
        block_df_full.loc[
            (block_df_full["meeting_id"] == t["meeting_id"]) & (block_df_full["turn_no"] == t["turn_no"]),
            "normalized_text"
        ].tolist()
    )
    if joined != t["raw_text"]:
        mismatches += 1
print(f"round-trip mismatches in {len(sample_check)}-turn sample: {mismatches}")
assert mismatches == 0

longest = turn_df_full.loc[turn_df_full["char_count"].idxmax()]
print(f"\nlongest turn: {longest['turn_no']}  meeting={longest['meeting_id']}  "
      f"pages {longest['page_start_no']}-{longest['page_end_no']}  "
      f"blocks {longest['block_start_no']} .. {longest['block_end_no']}  "
      f"char_count={longest['char_count']}")


round-trip mismatches in 2000-turn sample: 0

longest turn: 051381_T00005  meeting=051381  pages 2-6  blocks 051381_0002_0082 .. 051381_0006_0040  char_count=6729


## 11. 결론 (갱신) — `block_df`/`turn_df` FK 설계 확정

- `block_df` PK=`block_no` (`{meeting_id}_{page_no:04d}_{block_seq:04d}`), `block_text` 원문 보존,
  `turn_no` FK (PAGE_HEADER만 null).
- `turn_df` PK=`turn_no` (`{meeting_id}_T{seq:05d}`), `block_start_no`/`block_end_no` FK, `raw_text`는
  block_df 조인 결과와 **글자 단위로 100% 일치**함을 42개 문서 전체(65,590개 turn)에서 검증.
- 09절의 `build_segments()`(문자열만 합치는 방식)는 **폐기**하고, 이 `build_block_and_turn_df()`를
  다음 프로덕션 노트북의 기준 구현으로 삼는다.
